## Exercise — Regression with several descriptors

### Task 1 — Write and fit the solubility model

Copy this setup into your first cell. It uses the video's recipe with a different seed and 80 molecules. It builds the table, splits it into training and test sets, and defines the two scoring helpers.

```py
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
np.random.seed(1)
mw = np.random.normal(350, 80, 80)
logp = 0.01 * mw + np.random.normal(0, 0.8, 80)
logs = 0.16 - 0.63 * logp - 0.0062 * mw + np.random.normal(0, 0.4, 80)
df = pd.DataFrame({"mw": mw, "logp": logp, "logs": logs})
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
```

The model is

$$\widehat{\log S} = b_{\mathrm{MW}}\cdot \mathrm{MW} + b_{\log P}\cdot \log P + c$$

1. Write a function `solubility_model(desc, coef_mw, coef_logp, intercept)`. It takes the descriptor table as `desc` and reads the columns `"mw"` and `"logp"` by name.
2. Fit it with `curve_fit` on the training molecules, passing the training table as `x` and `train["logs"]` as `y`.
3. Unpack the result into `coef_mw`, `coef_logp` and `intercept_multi`.
4. Print the MW coefficient rounded to 4 decimals, the log P coefficient rounded to 3 decimals and the intercept rounded to 2 decimals.

Check whether your coefficients land close to the −0.0062 and −0.63 built into the data.


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
np.random.seed(1)
mw = np.random.normal(350, 80, 80)
logp = 0.01 * mw + np.random.normal(0, 0.8, 80)
logs = 0.16 - 0.63 * logp - 0.0062 * mw + np.random.normal(0, 0.4, 80)
df = pd.DataFrame({"mw": mw, "logp": logp, "logs": logs})
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def r_squared(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def solubility_model(desc, coef_mw, coef_logp, intercept):
    return coef_mw * desc["mw"] + coef_logp * desc["logp"] + intercept

popt, pcov = curve_fit(solubility_model, train, train["logs"])
coef_mw, coef_logp, intercept_multi = popt
print("coefficient MW:", round(coef_mw, 4))
print("coefficient logP:", round(coef_logp, 3))
print("intercept:", round(intercept_multi, 2))

### Task 2 — Paracetamol, and a heavier twin at the same log P

Paracetamol has the SMILES `CC(=O)Nc1ccc(O)cc1`.

1. Parse paracetamol with `Chem.MolFromSmiles` and check the result is not `None`.
2. Compute its molecular weight with `Descriptors.MolWt` and its log P with `Descriptors.MolLogP`. These are the descriptors you met in track 3 video 4.
3. Build a two-row DataFrame with columns `"mw"` and `"logp"`:
   - Row 1 is paracetamol as it is.
   - Row 2 is a hypothetical molecule that is 100 g/mol heavier but has **the same log P**.
4. Predict log S for both rows with `solubility_model` and your fitted coefficients, and print the predictions rounded to 3 decimals.
5. Print the difference (row 2 minus row 1), rounded to 3 decimals.
6. Print `100 * coef_mw`, rounded to 3 decimals.
7. End the cell with the molecule variable on its own line, so the structure is drawn.

Compare the two printed numbers and think about what that says a coefficient means.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import IPythonConsole
para = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")
if para is None:
    print("invalid SMILES: paracetamol")

para_mw = Descriptors.MolWt(para)
para_logp = Descriptors.MolLogP(para)
pair = pd.DataFrame({"mw": [para_mw, para_mw + 100], "logp": [para_logp, para_logp]})
pred_pair = solubility_model(pair, coef_mw, coef_logp, intercept_multi)
print(pred_pair.round(3))
print("difference:", round(pred_pair.iloc[1] - pred_pair.iloc[0], 3))
print("100 x coef_mw:", round(100 * coef_mw, 3))
para

### Task 3 — MW alone against MW + log P

1. Write the straight line from earlier videos, `line(mw, slope, intercept)`.
2. Fit it on the training molecules using molecular weight alone.
3. Unpack the result into `slope_mw` and `intercept_mw`, and print `slope_mw` rounded to 4 decimals. Compare it with `coef_mw` from Task 1.
4. You now have two tables, `train` and `test`. Choose the one that gives a fair comparison of the two models.
5. Predict log S on that table with both models.
6. Print four numbers, each rounded to 3 decimals:
   - R² for MW only
   - R² for MW + log P
   - MAE for MW only
   - MAE for MW + log P


In [ ]:
def line(mw, slope, intercept):
    return slope * mw + intercept

popt, pcov = curve_fit(line, train["mw"], train["logs"])
slope_mw, intercept_mw = popt
print("slope per g/mol, MW alone:", round(slope_mw, 4))
pred_single = line(test["mw"], slope_mw, intercept_mw)
pred_multi = solubility_model(test, coef_mw, coef_logp, intercept_multi)
print("test R2, MW only:", round(r_squared(test["logs"], pred_single), 3))
print("test R2, MW + logP:", round(r_squared(test["logs"], pred_multi), 3))
print("test MAE, MW only:", round(mae(test["logs"], pred_single), 3))
print("test MAE, MW + logP:", round(mae(test["logs"], pred_multi), 3))

### Task 4 — Predict, then check: a column of pure noise

Read the code below **without running it**. Write down your predictions first:

- (a) Is the training R² "with noise" higher than, lower than, or equal to the two-descriptor training R²?
- (b) Does the test R² "with noise" clearly improve on the two-descriptor test R²?

Then run the code and compare the output with your predictions.

```py
df["noise"] = np.random.normal(0, 1, len(df))
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def noisy_model(desc, coef_mw, coef_logp, coef_noise, intercept):
    return coef_mw * desc["mw"] + coef_logp * desc["logp"] + coef_noise * desc["noise"] + intercept

popt, pcov = curve_fit(noisy_model, train, train["logs"])
coef_mw3, coef_logp3, coef_noise, intercept3 = popt
fit_two = solubility_model(train, coef_mw, coef_logp, intercept_multi)
fit_three = noisy_model(train, coef_mw3, coef_logp3, coef_noise, intercept3)
print("train R2, two descriptors:", round(r_squared(train["logs"], fit_two), 4))
print("train R2, with noise:", round(r_squared(train["logs"], fit_three), 4))
pred_two = solubility_model(test, coef_mw, coef_logp, intercept_multi)
pred_three = noisy_model(test, coef_mw3, coef_logp3, coef_noise, intercept3)
print("test R2, two descriptors:", round(r_squared(test["logs"], pred_two), 4))
print("test R2, with noise:", round(r_squared(test["logs"], pred_three), 4))
```


In [ ]:
df["noise"] = np.random.normal(0, 1, len(df))
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

def noisy_model(desc, coef_mw, coef_logp, coef_noise, intercept):
    return coef_mw * desc["mw"] + coef_logp * desc["logp"] + coef_noise * desc["noise"] + intercept

popt, pcov = curve_fit(noisy_model, train, train["logs"])
coef_mw3, coef_logp3, coef_noise, intercept3 = popt
fit_two = solubility_model(train, coef_mw, coef_logp, intercept_multi)
fit_three = noisy_model(train, coef_mw3, coef_logp3, coef_noise, intercept3)
print("train R2, two descriptors:", round(r_squared(train["logs"], fit_two), 4))
print("train R2, with noise:", round(r_squared(train["logs"], fit_three), 4))
pred_two = solubility_model(test, coef_mw, coef_logp, intercept_multi)
pred_three = noisy_model(test, coef_mw3, coef_logp3, coef_noise, intercept3)
print("test R2, two descriptors:", round(r_squared(test["logs"], pred_two), 4))
print("test R2, with noise:", round(r_squared(test["logs"], pred_three), 4))

### Task 5 — Three molecules, three parameters

`solubility_model` has three parameters: two coefficients and an intercept.

1. Take only the first three training molecules: `tiny = train.iloc[0:3]`.
2. Before running anything, write down the training R² you expect when `solubility_model` is fitted on `tiny`.
3. Fit `solubility_model` on `tiny`.
4. Print its training R² on `tiny` and its R² on the full `test` set, both rounded to 4 decimals. You may see a warning that the covariance could not be estimated. That warning is itself a symptom of having too few molecules.
5. Use the rule of thumb of 10–20 molecules per descriptor. Print:
   - the number of molecules in the full training set
   - the most descriptors it supports at 10 molecules per descriptor
   - the most descriptors it supports at 20 molecules per descriptor

   Compute the last two with `//`.


In [ ]:
tiny = train.iloc[0:3]
popt, pcov = curve_fit(solubility_model, tiny, tiny["logs"])
coef_mw_t, coef_logp_t, intercept_t = popt
print("train R2, 3 molecules:", round(r_squared(tiny["logs"], solubility_model(tiny, coef_mw_t, coef_logp_t, intercept_t)), 4))
print("test R2, 3 molecules:", round(r_squared(test["logs"], solubility_model(test, coef_mw_t, coef_logp_t, intercept_t)), 4))
print("training molecules:", len(train))
print("max descriptors at 10 per descriptor:", len(train) // 10)
print("max descriptors at 20 per descriptor:", len(train) // 20)